# 10 — Export the experimental checkpoint for Streamlit
Run this CPU notebook once in Colab. Exports the completed Experiment 6 ResNet50 checkpoint, original evaluation reports, and one validation-reference fixture for local parity testing. It does not train, select a new candidate, or evaluate test images. The exported model still fails the SCC and macro-F1 requirements and has inadequate skin-type evidence. This is a development integration, not clinical deployment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
ROOT=Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
PACKAGE=ROOT/'Models'/'Selected'/'final_review_exp06_7fe97493900f'
metadata=json.loads((PACKAGE/'metadata.json').read_text())
assert metadata['class_names']==['BCC','Melanoma','SCC']
assert metadata['run_id']=='exp06_7fe97493900f'
assert metadata['preprocessing']=='resnet50'
assert metadata['decode']=='tf.io.decode_image channels=3; no EXIF orientation transform'
assert hashlib.sha256((PACKAGE/'model.keras').read_bytes()).hexdigest()==metadata['model_sha256']
reference=ROOT/'Reports'/'Training'/metadata['run_id']/metadata['model_name']
frame=pd.read_csv(reference/'validation_reference_images.csv')
with np.load(reference/'validation_reference.npz',allow_pickle=False) as archive:
    scores=archive['scores'].copy()
assert scores.shape[1]==3 and len(frame)==len(scores)
raw=Path(frame.iloc[0]['image_path']).read_bytes()
fixture=Path('/content/streamlit_reference_scores.npy')
np.save(fixture,scores[:1])
exports=ROOT/'Models'/'Exports';exports.mkdir(parents=True,exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
output=exports/f'streamlit_exp06_{stamp}.zip'
with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_STORED) as z:
    for name in ['model.keras','metadata.json','test_metrics.json','per_class_metrics.csv']:
        path=PACKAGE/name
        if not path.is_file():raise FileNotFoundError(path)
        z.write(path,arcname=name)
    z.writestr('reference_image.bin',raw)
    z.write(fixture,arcname='reference_scores.npy')
    z.writestr('reference_manifest.json',json.dumps(dict(
        image_sha256=hashlib.sha256(raw).hexdigest(),
        model_sha256=metadata['model_sha256'],
        note='One validation reference, not an independent performance evaluation'),indent=2))
print('Export saved to:',output)
print('Download this ZIP from Google Drive; extract its contents into app/models/active/.')
print('Keep these model files out of GitHub. Run scripts/verify_model_package.py locally before using the model.')
